<a href="https://colab.research.google.com/github/amilynestes1028/ds2002-fa26/blob/main/notebooks/05-walmart/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
# Filter product_name containing 'pop' (case-insensitive)
pop_tart_rows = cat[cat['product_name'].str.lower().str.contains('pop')]

# Print the count
print(f"Number of Pop-Tart rows: {len(pop_tart_rows)}\n")

# Print the matching rows
print(pop_tart_rows)

Number of Pop-Tart rows: 3

             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
# Separate Pop-Tarts into strawberry vs. non-strawberry
pop_rows = cat[cat['product_name'].str.lower().str.contains('pop')]

strawberry_pop = pop_rows[pop_rows['product_name'].str.lower().str.contains('strawberry')]
non_strawberry_pop = pop_rows[~pop_rows['product_name'].str.lower().str.contains('strawberry')]

print("--- Strawberry Pop-Tarts ---")
print(strawberry_pop[['sku', 'product_name']])

print("\n--- Non-Strawberry Pop-Tarts ---")
print(non_strawberry_pop)

--- Strawberry Pop-Tarts ---
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry

--- Non-Strawberry Pop-Tarts ---
Empty DataFrame
Columns: [sku, product_name, category]
Index: []


**Pop-Tart rows that are a different product:** There are no non-strawberry Pop-Tart rows present in the current dataset. All filtered Pop-Tart products are strawberry-flavored.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
# Choose one canonical SKU (e.g., the first one, 'PT-12')
canonical_sku = 'PT-12'

# Build a dictionary mapping each variant SKU to the canonical SKU
variant_skus = ['SKU#459812', 'POPTART-STRAW']
sku_mapping = {sku: canonical_sku for sku in variant_skus}

# Apply the mapping safely: use .map() followed by .fillna()
# so that unmapped SKUs remain as themselves instead of becoming NaN.
cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

# View the result
print(cat[['sku', 'canonical_sku', 'product_name']])

             sku canonical_sku          product_name
0          PT-12         PT-12  pop-tarts strawberry
1     SKU#459812         PT-12   Strawberry Pop-Tart
2  POPTART-STRAW         PT-12  Pop-Tarts Strawberry


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
# Show distinct values and counts before normalization
print("--- Before Normalization ---")
print(cat['category'].value_counts())

# Normalize category: strip whitespace and title-case
cat['normalized_category'] = cat['category'].str.strip().str.title()

# Show distinct values and counts after normalization
print("\n--- After Normalization ---")
print(cat['normalized_category'].value_counts())

--- Before Normalization ---
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

--- After Normalization ---
normalized_category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
import string
import re

# Define a function to normalize product names:
# 1. Lowercase
# 2. Strip punctuation (replace punctuation with spaces or remove)
# 3. Split into words, sort them alphabetically, and join back together
def normalize_product_name(name):
    if not isinstance(name, str):
        return ""
    # Convert to lowercase
    name = name.lower()
    # Replace punctuation with space so word boundaries are preserved
    name = re.sub(r'[^\w\s]', ' ', name)
    # Split into words, sort, and rejoin
    words = sorted(name.split())
    return ' '.join(words)

# Apply normalization
cat['normalized_name'] = cat['product_name'].apply(normalize_product_name)

# Group by normalized name and find groups holding more than one unique SKU
fragmented_groups = cat.groupby('normalized_name').filter(lambda x: x['sku'].nunique() > 1)

# Display the matching fragmented rows
print("--- Fragmented Product Groups (> 1 SKU) ---")
print(fragmented_groups[['sku', 'product_name', 'normalized_name']])

# Alternatively, summarize by group to see counts and SKUs
group_summary = cat.groupby('normalized_name').filter(lambda x: x['sku'].nunique() > 1) \
                   .groupby('normalized_name')['sku'].agg(['count', list])
print("\n--- Group Summary ---")
print(group_summary)

--- Fragmented Product Groups (> 1 SKU) ---
             sku          product_name       normalized_name
0          PT-12  pop-tarts strawberry  pop strawberry tarts
2  POPTART-STRAW  Pop-Tarts Strawberry  pop strawberry tarts

--- Group Summary ---
                      count                    list
normalized_name                                    
pop strawberry tarts      2  [PT-12, POPTART-STRAW]


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# TODO
# Save the original row count at the beginning or right before verification
original_row_count = len(cat)

# 1. Assert that there are no nulls in the canonical column
assert cat['canonical_sku'].isnull().sum() == 0, "Error: Null values found in canonical_sku!"

# 2. Assert that the total number of rows remains unchanged
assert len(cat) == original_row_count, f"Error: Row count changed from {original_row_count} to {len(cat)}!"

print("Verification passed: No nulls in canonical_sku and row count is fully preserved")

Verification passed: No nulls in canonical_sku and row count is fully preserved


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

1. Beyond Pop-Tarts, product families such as 24-pack bottled water and 4-pack AA batteries exhibit clear fragmentation across disparate SKUs and naming conventions (e.g., WTR-24 versus WATER24PK, and BATT-100 versus BAT-AA4). To group them, you would apply the programmatic normalization pipeline—lowercasing text, stripping punctuation, and sorting words—to surface matching groups, and then map all variant SKUs under a single designated canonical ID for each product.
2. Consolidation transitions from objective data cleaning into a strategic business decision when determining whether items with subtle variations—such as different pack sizes, minor supplier changes, or brand tiers—should be treated as interchangeable or distinct inventory. For instance, deciding whether a chips variety pack from Frito-Lay versus a generic supplier represent the same stockkeeping unit requires business judgment rather than simple text matching. To resolve this ambiguity, you would consult the Category Buyer or Merchandising Manager, as they manage the supplier contracts, pricing structures, and inventory strategies that dictate whether these items are truly identical to the business.